# Open-weight unlearning arm — Colab runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wrgr/socratic-scenarios/blob/claude/publishing-strategy-angle-yp7vor/experiments/unlearning/colab.ipynb)

Runs **Experiment 2** of the corpus-bounded-instruction paper: unlearn the
*alter-to-starboard* knowledge (COLREG Rule 14/15) from an open-weight LLM with
**SimNPO**, audit the removal, then score the base vs unlearned model on the
reference-optimal instrument — the 2×2 in `docs/novelty-and-positioning.md` §8.

### Before you run
1. **Runtime → Change runtime type → GPU.** An **A100** or **L4 (High-RAM)** is
   recommended for a 7–8B model in bf16 (~15 GB). On a **T4 (16 GB)** use a smaller
   model (set `MODEL = "Qwen/Qwen2.5-3B-Instruct"` below) or it may OOM.
2. Run the cells top to bottom.

The model steps run here on the GPU; scoring uses the repo's TypeScript instrument
against a locally-served OpenAI-compatible endpoint (Node is pre-installed on Colab).


## 1 · Confirm the GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → Change runtime type → GPU'

## 2 · Config + clone the repo

`BRANCH` defaults to the working branch that carries the SimNPO update. Switch it to
`main` once that work is merged.


In [ ]:
import os

REPO_URL = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH   = 'claude/publishing-strategy-angle-yp7vor'  # -> 'main' after merge
MODEL    = 'Qwen/Qwen2.5-7B-Instruct'   # T4? use 'Qwen/Qwen2.5-3B-Instruct'
METHOD   = 'simnpo'                     # simnpo (primary) | npo | ga
DTYPE    = 'bfloat16'                   # bf16 on GPU; float32 only on CPU

%cd /content
![ -d socratic-scenarios ] || git clone --depth 1 --branch $BRANCH $REPO_URL
REPO = '/content/socratic-scenarios'
ARM  = REPO + '/experiments/unlearning'
%cd $ARM

## 3 · Install Python deps

Colab GPU runtimes already ship a CUDA build of `torch`; we add the HF stack only.


In [ ]:
!pip -q install 'transformers>=4.40' 'peft>=0.11' 'accelerate>=0.30' 'safetensors>=0.4'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4 · Run the arm — build → unlearn (SimNPO) → audit

`run.sh` builds the forget/retain/audit sets, LoRA-unlearns the target rule, and prints
the removal audit (forget-set NLL ↑, retain-set NLL ~flat). A 7–8B run is ~minutes–hours
depending on the GPU.


In [ ]:
import os
env = dict(os.environ, MODEL=MODEL, METHOD=METHOD, DTYPE=DTYPE)
!cd $ARM && MODEL=$MODEL METHOD=$METHOD DTYPE=$DTYPE bash run.sh

## 5 · Score base vs unlearned on the instrument

> **Do I need an API key here? No.** This arm scores the **local** open-weight model you
> just unlearned — `serve.py` exposes it as an OpenAI-compatible endpoint on `localhost`,
> and the scorer uses a **dummy** `OPENAI_API_KEY=x` pointed at that local URL. No Gemini
> or OpenAI key is used or needed anywhere in this notebook.

Installs the repo's Node deps once, then serves each model (OpenAI-compatible) and runs
the leakage/diagnosis instrument against it. **Served sequentially** (base, then
unlearned) so a single GPU never holds two 7B models at once.

Read the reports as the §8 2×2: the **base** model should turn starboard from pretrained
priors even without the corpus (contamination baseline); the **unlearned** model should
fail the governed metric without the corpus and recover with it.


In [ ]:
# One-time: install Node deps for the tsx scorer (npx tsx scripts/colreg-leakage.ts).
!cd $REPO && npm install --no-audit --no-fund --loglevel=error

In [ ]:
import os, subprocess, time, urllib.request, json

def _wait_ready(port, timeout=600):
    body = json.dumps({'messages': [{'role': 'user', 'content': 'ping'}]}).encode()
    url = f'http://localhost:{port}/v1/chat/completions'
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            req = urllib.request.Request(url, data=body, headers={'Content-Type': 'application/json'})
            urllib.request.urlopen(req, timeout=10)
            return True
        except urllib.error.HTTPError:
            return True   # server answered (any HTTP status = up)
        except Exception:
            time.sleep(3)
    return False

def serve_and_score(label, adapter=None, port=8000):
    cmd = ['python', 'serve.py', '--model', MODEL, '--dtype', DTYPE, '--port', str(port)]
    if adapter:
        cmd += ['--adapter', adapter]
    print(f'\n===== {label}: starting server ({" ".join(cmd)}) =====')
    srv = subprocess.Popen(cmd, cwd=ARM)
    try:
        if not _wait_ready(port):
            raise RuntimeError('server did not become ready')
        env = dict(os.environ, OPENAI_API_KEY='x',
                   OPENAI_BASE_URL=f'http://localhost:{port}/v1', OPENAI_MODEL='local')
        r = subprocess.run(['npm', 'run', 'colreg:leakage'], cwd=REPO, env=env,
                           capture_output=True, text=True)
        print(r.stdout[-6000:])
        if r.returncode != 0:
            print('--- stderr ---'); print(r.stderr[-2000:])
    finally:
        srv.terminate()
        try:
            srv.wait(timeout=15)
        except Exception:
            srv.kill()

serve_and_score('BASE (not unlearned)', adapter=None, port=8000)
serve_and_score('UNLEARNED', adapter='out/unlearned', port=8001)

## Done

That's the full arm: unlearn → audit → score base-vs-unlearned on the instrument. Read the
two reports as the §8 2×2. No API key was needed anywhere — the scorer talked only to the
local model.
